In [1]:
import sys
import os

import pandas as pd


In [2]:
# Get project root
project_root = os.path.abspath("..")

# Add to Python path
sys.path.append(project_root)

print(project_root)

/home/chaithanaya/Documents/ai-route-optimizer


In [4]:
from services.google_routes_service import *
from services.route_optimizer import *

In [5]:
df = pd.read_csv("../data/historical_trips.csv")

df.head()

,trip_id,driver_id,date,stop_number,location_id,store_name,latitude,longitude,arrival_time,visit_duration_min,distance_to_next_km,travel_time_to_next_min
0,1,D1,2026-04-01,1,L1,Ratnadeep Supermarket,17.455771,78.212505,09:00,14,10.82,16.2
1,1,D1,2026-04-01,2,L25,IKEA Hyderabad,17.360466,78.233094,09:44,37,44.79,67.2
2,1,D1,2026-04-01,3,L40,Apollo Health City,17.588755,78.580981,11:34,18,33.07,49.6
3,1,D1,2026-04-01,4,L31,Cafe Niloufer,17.598930,78.269166,12:48,25,37.58,56.4
4,1,D1,2026-04-01,5,L33,Pista House,17.261137,78.279991,14:21,25,20.01,30.0


In [12]:
trip_ids = df["trip_id"].unique()[:10]

trip_ids

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10])

In [13]:
optimized_results = []

In [14]:
for trip_id in trip_ids:

    # Filter one trip
    trip_df = df[df["trip_id"] == trip_id]

    # Extract locations
    locations = list(
        zip(
            trip_df["latitude"],
            trip_df["longitude"]
        )
    )

    # Store names
    store_names = trip_df["store_name"].tolist()

    print(f"\nProcessing Trip: {trip_id}")
    print(store_names)


Processing Trip: 1
['Ratnadeep Supermarket', 'IKEA Hyderabad', 'Apollo Health City', 'Cafe Niloufer', 'Pista House', 'Apollo Pharmacy']

Processing Trip: 2
['Decathlon Madhapur', 'South India Shopping Mall', 'Joyalukkas']

Processing Trip: 3
['Karachi Bakery', 'Naturals Salon', 'Roastery Coffee House', 'Bata Shoe Store', 'Joyalukkas']

Processing Trip: 4
['Karachi Bakery', 'Bajaj Electronics', 'Lalitha Jewellery', 'Kalanikethan Kukatpally', 'Inorbit Mall Shoppers Stop', 'Lakme Salon', 'Roastery Coffee House']

Processing Trip: 5
['Reliance Smart Jubilee', 'Bajaj Electronics', 'Joyalukkas', 'IKEA Hyderabad']

Processing Trip: 6
['Decathlon Madhapur', 'South India Shopping Mall', 'Joyalukkas']

Processing Trip: 7
['Decathlon Madhapur', 'South India Shopping Mall', 'Joyalukkas']

Processing Trip: 8
['Karachi Bakery', 'Bajaj Electronics', 'Lalitha Jewellery', 'Kalanikethan Kukatpally', 'Inorbit Mall Shoppers Stop', 'Lakme Salon', 'Roastery Coffee House']

Processing Trip: 9
['Decathlon Ma

In [15]:
for trip_id in trip_ids:

    # Filter one trip
    trip_df = df[df["trip_id"] == trip_id]

    # Extract locations
    locations = list(
        zip(
            trip_df["latitude"],
            trip_df["longitude"]
        )
    )
    print(locations)

[(17.455771, 78.212505), (17.360466, 78.233094), (17.588755, 78.580981), (17.59893, 78.269166), (17.261137, 78.279991), (17.373906, 78.426862)]
[(17.57878, 78.242827), (17.454274, 78.382416), (17.435323, 78.315057)]
[(17.236364, 78.223558), (17.261136, 78.581255), (17.598449, 78.464557), (17.571565, 78.577633), (17.435323, 78.315057)]
[(17.236364, 78.223558), (17.443652, 78.285569), (17.228343, 78.319002), (17.546593, 78.390063), (17.237098, 78.248358), (17.439578, 78.444303), (17.598449, 78.464557)]
[(17.287455, 78.452678), (17.443652, 78.285569), (17.435323, 78.315057), (17.360466, 78.233094)]
[(17.57878, 78.242827), (17.454274, 78.382416), (17.435323, 78.315057)]
[(17.57878, 78.242827), (17.454274, 78.382416), (17.435323, 78.315057)]
[(17.236364, 78.223558), (17.443652, 78.285569), (17.228343, 78.319002), (17.546593, 78.390063), (17.237098, 78.248358), (17.439578, 78.444303), (17.598449, 78.464557)]
[(17.57878, 78.242827), (17.454274, 78.382416), (17.435323, 78.315057)]
[(17.57878, 

In [16]:
for trip_id in trip_ids:

    # Filter one trip
    trip_df = df[df["trip_id"] == trip_id]

    # Extract locations
    locations = list(
        zip(
            trip_df["latitude"],
            trip_df["longitude"]
        )
    )
    store_names = trip_df["store_name"].tolist()

    matrix_response = get_route_matrix(locations)

    duration_matrix = extract_duration_matrix(
        matrix_response,
        len(locations)
    )
    optimized = nearest_neighbor_optimizer(
        duration_matrix
    )
    optimized_store_order = [
        store_names[i]
        for i in optimized["optimized_route"]
    ]
    optimized_results.append({

        "trip_id": trip_id,

        "original_route":
            " -> ".join(store_names),

        "optimized_route":
            " -> ".join(
                optimized_store_order
            ),

        "optimized_duration_min":
            optimized["total_duration_min"]
    })

In [17]:
optimized_df = pd.DataFrame(
    optimized_results
)

optimized_df.head()

,trip_id,original_route,optimized_route,optimized_duration_min
0,1,Ratnadeep Supermarket -> IKEA Hyderabad -> Apo...,Ratnadeep Supermarket -> Cafe Niloufer -> Apol...,281.05
1,2,Decathlon Madhapur -> South India Shopping Mal...,Decathlon Madhapur -> Joyalukkas -> South Indi...,77.51
2,3,Karachi Bakery -> Naturals Salon -> Roastery C...,Karachi Bakery -> Naturals Salon -> Bata Shoe ...,206.78
3,4,Karachi Bakery -> Bajaj Electronics -> Lalitha...,Karachi Bakery -> Inorbit Mall Shoppers Stop -...,211.56
4,5,Reliance Smart Jubilee -> Bajaj Electronics ->...,Reliance Smart Jubilee -> Joyalukkas -> Bajaj ...,129.54


In [18]:
optimized_df.to_csv(
    "../data/optimized_routes_demo.csv",
    index=False
)

print("Optimized routes saved!")

Optimized routes saved!
